# Exercises — Strategy optimization and benchmarking

[DataCamp exercise](https://campus.datacamp.com/courses/financial-trading-in-python/trading-strategies?ex=12) · see `Notes.md` in this folder for the summary.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project's data folder regardless of where this notebook runs from
DATA = next(p / "course materials" / "data"
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "course materials" / "data").is_dir())

def load(name):
    """Load an OHLCV CSV with a parsed DatetimeIndex."""
    return pd.read_csv(DATA / name, index_col="Date", parse_dates=True)

def price(name, col, year=None):
    """Single-asset price DataFrame (column = `col`) for use with bt."""
    df = load(name)
    if year:
        df = df[df.index.year == year]
    return df["Close"].rename(col).to_frame()


### Optimize the SMA lookback, then benchmark vs buy-and-hold

In [ ]:
import bt
import talib

data = price("AMZN-stock-data.csv", "AMZN", year=2020)

def signal_strategy(data, name, period):
    px = data.iloc[:, 0]
    sma = talib.SMA(px, timeperiod=period)
    signal = pd.DataFrame(px.values > sma.values, index=data.index, columns=data.columns)
    strat = bt.Strategy(name, [bt.algos.SelectWhere(signal),
                               bt.algos.WeighEqually(), bt.algos.Rebalance()])
    return bt.Backtest(strat, data)

def buy_and_hold(data, name):
    strat = bt.Strategy(name, [bt.algos.RunOnce(), bt.algos.SelectAll(),
                               bt.algos.WeighEqually(), bt.algos.Rebalance()])
    return bt.Backtest(strat, data)

tests = [signal_strategy(data, f"SMA{n}", n) for n in (10, 20, 50)]
tests.append(buy_and_hold(data, "Buy & hold"))

bt_results = bt.run(*tests)
bt_results.plot(title="SMA optimization vs benchmark")
plt.show()
bt_results.display()